# 3.4.6 Handling Imbalanced Data

class_weight='balanced', threshold tuning, StratifiedKFold, PR-AUC.

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score,average_precision_score,confusion_matrix
from sklearn.pipeline import make_pipeline
import warnings; warnings.filterwarnings('ignore')
h=fetch_california_housing(); X,y_c=h.data,h.target
y=(y_c>=np.percentile(y_c,90)).astype(int)
print(f'Class 1: {y.mean()*100:.1f}%  (n={y.sum()})')

In [ ]:
Xt,Xv,yt,yv=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
for cw in [None,'balanced']:
    p=make_pipeline(StandardScaler(),LogisticRegression(class_weight=cw,max_iter=1000))
    p.fit(Xt,yt); pr=p.predict_proba(Xv)[:,1]
    tn,fp,fn,tp=confusion_matrix(yv,p.predict(Xv)).ravel()
    print(f'cw={str(cw):10s} ROC={roc_auc_score(yv,pr):.4f} PR-AUC={average_precision_score(yv,pr):.4f} Recall={tp/(tp+fn):.3f}')

In [ ]:
p=make_pipeline(StandardScaler(),LogisticRegression(class_weight='balanced',max_iter=1000))
p.fit(Xt,yt); probs=p.predict_proba(Xv)[:,1]
print(f'{"Thresh":>8} {"Prec":>8} {"Rec":>8} {"F1":>8}')
for t in [.3,.4,.5,.6,.7]:
    pred=(probs>=t).astype(int)
    if pred.sum()==0: print(f'{t:8.1f} no positives'); continue
    tn,fp,fn,tp=confusion_matrix(yv,pred).ravel()
    pr2=tp/(tp+fp) if tp+fp else 0; rec=tp/(tp+fn) if tp+fn else 0
    f1=2*pr2*rec/(pr2+rec) if pr2+rec else 0
    print(f'{t:8.1f} {pr2:8.3f} {rec:8.3f} {f1:8.3f}')

In [ ]:
from sklearn.model_selection import StratifiedKFold,cross_val_score
skf=StratifiedKFold(5,shuffle=True,random_state=42)
p=make_pipeline(StandardScaler(),LogisticRegression(class_weight='balanced',max_iter=1000))
sc=cross_val_score(p,X,y,cv=skf,scoring='average_precision')
print(f'Stratified 5-fold PR-AUC: {sc.mean():.4f} ± {sc.std():.4f}')